# 七天终点状态审计：只读结果核对
在本 notebook 所在目录运行。此检查不训练、不预测，也不改写任何数据。
这些类别是预定规则下的证据分层，不是真实搁浅发生率。逐窗口计算与独立检查见 code/run.py 和 code/audit.py。

In [1]:
from pathlib import Path
import pandas as pd
root = Path.cwd()
f = pd.read_parquet(root / "results/endpoint_states.parquet")
s = pd.read_csv(root / "results/summary.csv")
assert not f.duplicated(["id", "origin_source_obs", "horizon_days"]).any()
assert f.id.nunique() == 383
for scope, mask in [("all_risk", f.risk_set), ("frozen_used", f.used_by_original)]:
    for cohort, g in f[mask].groupby("cohort"):
        expected = s[(s.scope == scope) & (s.cohort == cohort) & (s.version == "primary_5km")]
        assert expected.pairs.sum() == len(g)
        for row in expected.itertuples(index=False):
            selected = g[g.state == row.state]
            assert len(selected) == row.pairs
            assert selected.id.nunique() == row.ids
print(s[s.version == "primary_5km"][["scope", "cohort", "state", "pairs", "ids", "pair_fraction"]].to_string(index=False))


      scope     cohort              state  pairs  ids  pair_fraction
   all_risk      train grounding_evidence      0    0       0.000000
   all_risk      train  drifting_evidence 245317  332       0.882432
   all_risk      train          uncertain  24793  249       0.089183
   all_risk      train   endpoint_missing   7891   67       0.028385
   all_risk validation grounding_evidence      0    0       0.000000
   all_risk validation  drifting_evidence  30621   45       0.876764
   all_risk validation          uncertain   3507   38       0.100415
   all_risk validation   endpoint_missing    797    7       0.022820
frozen_used      train grounding_evidence      0    0       0.000000
frozen_used      train  drifting_evidence 243114  332       0.916053
frozen_used      train          uncertain  22279  249       0.083947
frozen_used      train   endpoint_missing      0    0       0.000000
frozen_used validation grounding_evidence      0    0       0.000000
frozen_used validation  drifting_e